# DocGuard-VLM: LoRA fine-tuning of Qwen2-VL-2B for document OCR + forgery detection

Runs on a free Colab **T4** GPU via [unsloth](https://github.com/unslothai/unsloth) QLoRA.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

**Setup:** clone the `docguard-vlm` repo (code only, ~a few hundred KB) and regenerate the
dataset *inside Colab* rather than uploading it — `data/processed/` is ~1.5GB of images and
isn't committed to git:
```
!git clone <your-repo-url> docguard-vlm
%cd docguard-vlm
!pip install -q datasets huggingface_hub tqdm pillow opencv-python-headless
!PYTHONPATH=src python src/data_gen/build_dataset.py --out-dir data/processed \
    --n-train 350 --n-test-clean 60 --n-test-adv 60
```
Takes a couple of minutes on Colab's network/CPU. Same seed as local, so the split is
reproducible if you want to sanity-check against images you already inspected locally.

In [ ]:
!pip install -q unsloth trl==0.12.1 peft accelerate bitsandbytes pillow

In [ ]:
DATA_ROOT = "data/processed"  # adjust if you cloned/uploaded elsewhere
OUTPUT_DIR = "outputs/qwen2vl-2b-docguard-lora"
BASE_MODEL = "unsloth/Qwen2-VL-2B-Instruct"
MAX_STEPS = None       # set an int to cap steps for a quick run; None = full epochs
NUM_EPOCHS = 2
LORA_R = 16
LORA_ALPHA = 16

In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

## Load DocGuard-VLM JSONL and convert to unsloth's vision chat format

Each record already stores an instruction/response pair (OCR field-extraction or forgery-verdict) built by `src/data_gen/build_dataset.py`.

In [ ]:
import json, os
from PIL import Image

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def to_conversation_sample(record, data_root):
    img = Image.open(os.path.join(data_root, record["image"])).convert("RGB")
    messages = []
    for turn in record["conversations"]:
        content = []
        for c in turn["content"]:
            if c["type"] == "image":
                content.append({"type": "image", "image": img})
            else:
                content.append({"type": "text", "text": c["text"]})
        messages.append({"role": turn["role"], "content": content})
    return {"messages": messages}

train_records = load_jsonl(os.path.join(DATA_ROOT, "train.jsonl"))
train_dataset = [to_conversation_sample(r, DATA_ROOT) for r in train_records]
print(f"train samples: {len(train_dataset)}")
print(train_dataset[0]["messages"][0]["content"][1]["text"])
print(train_dataset[0]["messages"][1]["content"][0]["text"])

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        num_train_epochs=NUM_EPOCHS,
        max_steps=MAX_STEPS if MAX_STEPS else -1,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
        # vision-specific: unsloth's collator needs these off so it can pack image+text itself
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=2048,
    ),
)

trainer_stats = trainer.train()

In [ ]:
# Save the LoRA adapter (small, a few hundred MB) -- this is what you download
# and hand to src/eval/evaluate.py for scoring.
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("saved to", OUTPUT_DIR)

## Quick smoke test
Run one clean and one adversarial example through the fine-tuned model before doing the full evaluation harness.

In [ ]:
FastVisionModel.for_inference(model)

test_records = load_jsonl(os.path.join(DATA_ROOT, "test_clean.jsonl"))
sample = next(r for r in test_records if r["task"] == "forgery")
img = Image.open(os.path.join(DATA_ROOT, sample["image"])).convert("RGB")
instruction = sample["conversations"][0]["content"][1]["text"]

messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction}]}]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(img, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

out = model.generate(**inputs, max_new_tokens=128, use_cache=True, temperature=0.2)
print("GROUND TRUTH:", sample["conversations"][1]["content"][0]["text"])
print("PREDICTION  :", tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Next: run the full evaluation harness
Download `outputs/qwen2vl-2b-docguard-lora/` and run locally, or continue in this notebook:
```
!python src/eval/evaluate.py \
  --data-root data/processed \
  --adapter outputs/qwen2vl-2b-docguard-lora \
  --base-model unsloth/Qwen2-VL-2B-Instruct \
  --out results/eval_results.json
```
This scores zero-shot baseline vs. fine-tuned, on both `test_clean.jsonl` and `test_adversarial.jsonl`.